In [12]:
import pandas as pd
import re

In [7]:
data1 = pd.read_csv(r"C:\\Personal-space\\projects\\source1_naukri_applicants.csv")
data1.sample(5)

,Full Name,Email,Phone,City,Experience (Years),Current CTC,Applied Date,Skills
24,Sneha Mishra,sneha.mishra14@mailtest.example.org,9000000229,pune,1.4,410629.0,15-06-2026,"MySQL, Docker, Selenium, Pandas"
26,Priya Saxena,priya.saxena61@example.com,919000000231,Delhi,5.0,6.6,08/16/2026,"Python, MySQL, Zapier, React"
5,Sahil Malhotra,sahil.malhotra1@example.in,919000000143,Noida,1.7,806661.0,07/13/2026,"FastAPI, Docker, MySQL, Zapier, Pandas, REST APIs"
25,Nikhil Chopra,alt.nikhil.chopra70@example.com,9000000103,NOIDA,0.8,7.8,07/03/2026,"Pandas, SQL, n8n"
41,Neha Bhatia,neha.bhatia60@mailtest.example.org,9000000273,Pune,4.2,694306.0,22 Jul 2026,"FastAPI, Zapier, JavaScript, Selenium"


In [9]:
data2 = pd.read_csv(r"C:\\Personal-space\\projects\\source2_gig_workers.csv")
data2.sample(5)

,email_id,worker_name,rate,location,status,skill_tags
21,vikram.mehta6@example.com,Vikram Mehta,22k/month,pune,ACTIVE,"rest apis, python, langchain"
17,manish.bhatia3@example.com,Manish Bhatia,73k/month,Noida,ACTIVE,"pandas, docker, javascript, react, rest apis, ..."
6,meera.bhatia52@mailtest.example.org,Meera Bhatia,330/hr,New Delhi,active,"langchain, docker, mysql, python, sql, react"
11,VARUN.SAXENA21@EXAMPLE.IN,Varun Saxena,917/hr,gurugram,Inactive,"mongodb, zapier, sql, langchain, mysql, rest apis"
7,vikram.saxena60@example.com,Vikram Saxena,843/hr,Gurgaon,Active,"selenium, web scraping, react, docker, sql, fa..."


In [10]:
data3 = pd.read_csv(r"C:\\Personal-space\\projects\\source3_cbnexus_contacts.csv")
data3.sample(5)

,Name,Phone Number,City,Verified,Projects Completed
12,KARAN BHATIA,9000000211,NOIDA,yes,2
18,Vikram Saxena,9000000113,Gurgaon,No,15
21,VARUN SAXENA,919000000170,GURGAON,Y,1
11,Tanvi Agarwal,919000000148,Pune,N,3
8,Shreya Gupta,+91-9000000227,NOIDA,yes,13


In [11]:
data1.info(), data2.info(), data3.info()

<class 'pandas.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Full Name           42 non-null     str    
 1   Email               42 non-null     str    
 2   Phone               42 non-null     int64  
 3   City                42 non-null     str    
 4   Experience (Years)  42 non-null     float64
 5   Current CTC         42 non-null     float64
 6   Applied Date        42 non-null     str    
 7   Skills              42 non-null     str    
dtypes: float64(2), int64(1), str(5)
memory usage: 2.8 KB
<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   email_id     31 non-null     str  
 1   worker_name  31 non-null     str  
 2   rate         31 non-null     str  
 3   location     31 non-null     str  
 4   status       31 non-null 

(None, None, None)

# Email

In [13]:
EMAIL_PATTERN = re.compile(
    r"^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$"
)

def normalize_email(value):
    """Normalize a single email value."""
    if pd.isna(value):
        return None

    value = str(value).strip().lower()

    if not value:
        return None

    return value


def is_valid_email(value):
    """Check whether a value is a valid email."""
    if not value:
        return False

    return bool(EMAIL_PATTERN.match(value))


def find_email(row, email_column="Email"):
    """
    Find and normalize an email for one row.

    1. Check the expected email column first.
    2. If invalid/missing, search every other column.
    3. Return the first valid email found.
    """

    # Check the expected email column first
    if email_column in row.index:
        email = normalize_email(row[email_column])

        if is_valid_email(email):
            return email

    # Search all other columns
    for column in row.index:

        # Don't check the email column again
        if column == email_column:
            continue

        value = normalize_email(row[column])

        if is_valid_email(value):
            return value

    return None


def process_emails(df, email_column="Email"):
    """Find, normalize and validate emails for the entire DataFrame."""

    df = df.copy()

    df["email"] = df.apply(
        lambda row: find_email(row, email_column),
        axis=1
    )

    # df["email_valid"] = df["email"].notna()

    return df

# Phone

In [15]:
def normalize_phone(value):
    if pd.isna(value):
        return None

    value = str(value).strip()

    # Allow only a trailing .0 caused by pandas/Excel
    if re.fullmatch(r"\d+\.0", value):
        value = value[:-2]

    # If decimal occurs anywhere else, reject it
    elif "." in value:
        return None

    # Remove non-digit characters
    digits = re.sub(r"\D", "", value)

    # Handle +91 / 91
    if digits.startswith("91") and len(digits) == 12:
        digits = digits[2:]

    # Must be exactly 10 digits
    if len(digits) != 10:
        return None

    # Indian mobile number
    if digits[0] not in "6789":
        return None

    return f"+91-{digits}"


def find_phone(row, phone_column="Phone"):
    """
    Find a valid phone number in a row.

    Priority:
    1. Phone column
    2. Other columns
    3. Return None if no valid phone is found
    """

    # --------------------------------------------------
    # 1. Check Phone column first
    # --------------------------------------------------
    if phone_column in row.index:

        phone = normalize_phone(row[phone_column])

        if phone is not None:
            return phone

    # --------------------------------------------------
    # 2. Search all other columns
    # --------------------------------------------------
    for column in row.index:

        if column == phone_column:
            continue

        phone = normalize_phone(row[column])

        if phone is not None:
            return phone


    return None

In [16]:
def normalize_data1(data):
    # Normalize data1 (source1_naukri_applicants.csv)

    name = data['Full Name'].str.lower().str.strip()
    email = process_emails(data, email_column="Email")['email']

    phone = data["Phone"].apply(normalize_phone)

    # print("Normalized data1:")
    # print(f"Name: {name}")
    # print(f"Email: {email}")

    print(f"Phone: {phone}")

    

    # phone = data['Phone'].str.strip()
    # city = data['City'].str.strip()
    # experience = data['Experience'].str.strip()
    # ctc = data['Current CTC'].str.strip()
    # applied_date = data['Applied Date'].str.strip()
    # skills = data['Skills'].str.strip()

    # return pd.DataFrame({
    #     'name': name,
    #     'email': email,
    #     'phone': phone,
    #     'city': city,
    #     'experience': experience,
    #     'ctc': ctc,
    #     'applied_date': applied_date,
    #     'skills': skills
    # })

normalize_data1(data1)


Phone: 0     +91-9000000254
1     +91-9000000237
2     +91-9000000287
3     +91-9000000113
4     +91-9000000288
5     +91-9000000143
6     +91-9000000227
7     +91-9000000138
8     +91-9000000222
9     +91-9000000167
10    +91-9000000260
11    +91-9000000114
12    +91-9000000145
13    +91-9000000211
14    +91-9000000146
15    +91-9000000106
16    +91-9000000223
17    +91-9000000263
18    +91-9000000131
19    +91-9000000177
20    +91-9000000162
21    +91-9000000116
22    +91-9000000271
23    +91-9000000294
24    +91-9000000229
25    +91-9000000103
26    +91-9000000231
27    +91-9000000170
28    +91-9000000133
29    +91-9000000294
30    +91-9000000268
31    +91-9000000296
32    +91-9000000137
33    +91-9000000202
34    +91-9000000165
35    +91-9000000103
36    +91-9000000148
37    +91-9000000104
38    +91-9000000295
39    +91-9000000212
40    +91-9000000259
41    +91-9000000273
Name: Phone, dtype: str
